## Parameter Recovery

Check identifiability.

1. Sample *known* true parameters
2. Simulate data from those parameters
3. Fit the model to recover parameters
4. Correlate true vs recovered

High correlation means the model's parameters are identifiable.

In [ ]:
mean_pr = [0.3, -5.0]
cov_pr  = [[0.01, 0.0], [0.0, 0.5]]
n_sets  = 50

print("Parameter recovery (5 repetitions):")
for rep in range(5):
    sampled = multivariate_normal.rvs(mean=mean_pr, cov=cov_pr, size=n_sets)
    # Resample nonsensical values
    valid = (sampled[:,0]>0) & (sampled[:,0]<1) & (sampled[:,1]<0)
    while not valid.all():
        n_bad = (~valid).sum()
        new = multivariate_normal.rvs(mean=mean_pr, cov=cov_pr, size=max(n_bad, 2))
        sampled[~valid] = new[:n_bad] if n_bad > 1 else new[:1]
        valid = (sampled[:,0]>0) & (sampled[:,0]<1) & (sampled[:,1]<0)

    recovered = np.array([fit_model(nll_model1, *simulate_model1(s[0], s[1]), x0=[0.3,-8]).x
                          for s in sampled])

    r_a, _ = pearsonr(sampled[:,0], recovered[:,0])
    r_b, _ = pearsonr(sampled[:,1], recovered[:,1])
    print(f"  Rep {rep+1}: r(α) = {r_a:.4f},  r(β) = {r_b:.4f}")

    if rep == 0:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].scatter(sampled[:,0], recovered[:,0], alpha=0.6, edgecolors="w")
        axes[0].plot([0,1],[0,1], "k--"); axes[0].set_xlabel("True α"); axes[0].set_ylabel("Recovered α")
        axes[0].set_title(f"α  (r = {r_a:.3f})")
        axes[1].scatter(sampled[:,1], recovered[:,1], alpha=0.6, edgecolors="w")
        lo,hi = sampled[:,1].min()-0.5, sampled[:,1].max()+0.5
        axes[1].plot([lo,hi],[lo,hi], "k--"); axes[1].set_xlabel("True β"); axes[1].set_ylabel("Recovered β")
        axes[1].set_title(f"β  (r = {r_b:.3f})")
        plt.tight_layout(); plt.savefig("figures/task_g.png"); plt.show()